# LORA replaces the expensive weight with a row rank decomposition.
 - r: Think of r as the learning capacity you attach to DistilBERT.  Bigger r → the add‑on can learn more patterns.- Smaller r → lighter, faster, but learns less. 8 is a medium size not too big or small.
 - lora_alpha: How strongly the add in speak to the model. Higher alpha → the LoRA add‑on speaks louder.Lower alpha → it whispers.Setting it to 16 means: “Let the LoRA module influence the model, but not overpower it.
 - lora_dropout: Dropout is like telling the model:“Don’t rely too heavily on any one part of the LoRA module.Learn more general patterns.”A value of 0.1 means: “Randomly turn off 10% of LoRA neurons during training.” This keeps the model from memorizing the training data.
 - bias: Only train the LoRA add‑on. Don’t modify the original DistilBERT biases
 - task_type: Inject your tiny learning modules into the parts of the model that matter for sentence classification.It ensures LoRA modifies the right layers (attention layers)



 Putting it all together in one simple analogy
    Imagine DistilBERT is a big factory machine that already knows a lot about language.
    LoRA adds small detachable gears to help it learn a new task (sentiment classification) without rebuilding the whole machine.
- r=8 → size of the new gears
- alpha=16 → how much those gears affect the machine
- dropout=0.1 → occasionally turning gears off so they don’t overfit
- bias="none" → don’t touch the original machine parts
- task_type="SEQ_CLS" → attach gears to the parts used for classificatio
- target_modules= Use to tell LORA where to inject. LoRA into the query and value projection layers of attention.These are the standard LoRA          injection points for DistilBERT.This gives excellent performance with minimal parameters.











In [98]:
from dataclasses import dataclass, asdict
from datetime import datetime
import torch
import torch
import random
import numpy as np
from sklearn.metrics import accuracy_score, f1_score
from datasets import load_dataset, DatasetDict
from peft import (
    get_peft_model,
    LoraConfig,
    IA3Config,
    AdaLoraConfig,
    PrefixTuningConfig,
    PromptTuningConfig,
    TaskType,
    AutoPeftModelForSequenceClassification
)
from transformers import (
    AutoModelForSequenceClassification, 
    AutoTokenizer, 
    TrainingArguments, 
    Trainer,
    DataCollatorWithPadding
)
import os
import json
from datetime import datetime
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix
import time


In [99]:

SUPPORTED_MODELS = {
    "bert-base-uncased": "bert",
    "roberta-base": "roberta",
    "distilbert-base-uncased": "distilbert",
}

@dataclass
class ExperimentConfig:
    model_name: str = "bert-base-uncased"
    peft_method: str = "lora"   # lora | ia3 | adalora | prefix | prompt | freeze
    learning_rate: float = 2e-4
    batch_size: int = 16    # Lower batch size is better for CPU RAM
    num_epochs: int = 1
    output_root: str = "./outputs"
    seed: int = 42
    max_length: int = 128
    num_labels: int = 2
    dataset_name: str = "imdb"
    _exp_name: str = None

    def device(self):
        return torch.device("cuda" if torch.cuda.is_available() else "cpu")

    def to_dict(self):
        return asdict(self)

    def experiment_name(self):
        # generate ONCE, reuse forever
        if self._exp_name is None:
            ts = datetime.now().strftime("%Y%m%d_%H%M%S")
            self._exp_name = f"{self.peft_method}_{ts}"
        return self._exp_name

    def output_dir(self):
        return f"{self.output_root}/{self.experiment_name()}"


In [ ]:

def build_model(config: ExperimentConfig, total_steps: int):
    model = AutoModelForSequenceClassification.from_pretrained(
        config.model_name,
        num_labels=config.num_labels
    )
    model = apply_peft(model, config, total_steps)
    return model.to(config.device())


#This function automatically detects the correct attention projection layers (query/value) for LoRA injection, 
# making your PEFT pipeline work across many different Transformer architectures without manual configuration
def guess_target_modules(model):
    names = []
    for name, _ in model.named_modules():
        if any(k in name for k in ["query", "value", "q_lin", "v_lin"]):
            leaf = name.split(".")[-1]
            names.append(leaf)
    return list(set(names)) or ["query", "value"]


#freezes the entire model, optionally unfreezes the last N encoder layers, and always keeps the classifier head trainable
def apply_freeze(model, unfreeze_last_n=0, total_steps=None):

    # 1. Freeze everything
    for _, p in model.named_parameters():
        p.requires_grad = False

    # 2. Detect encoder layers
    if hasattr(model, "bert"):
        encoder = model.bert.encoder.layer
    elif hasattr(model, "roberta"):
        encoder = model.roberta.encoder.layer
    elif hasattr(model, "distilbert"):
        encoder = model.distilbert.transformer.layer
    else:
        encoder = None

    # 3. Unfreeze last N encoder layers
    if encoder is not None and unfreeze_last_n > 0:
        for layer in encoder[-unfreeze_last_n:]:
            for _, p in layer.named_parameters():
                p.requires_grad = True

    # 4. Unfreeze classifier using your helper
    classifier = find_classifier(model)
    if classifier is not None:
        for p in classifier.parameters():
            p.requires_grad = True

    return model

def apply_peft(model, config: ExperimentConfig, total_steps=None):
    #This line tells the PEFT library what kind of task the training model will be used.
    task_type = TaskType.SEQ_CLS

    if config.peft_method == "freeze":
        return apply_freeze(model, unfreeze_last_n=1)

    target_modules = guess_target_modules(model)

    if config.peft_method == "lora":
        peft_config = LoraConfig(
            task_type=task_type,
            r=8,
            lora_alpha=16,
            target_modules=target_modules,
            lora_dropout=0.1,
        )
    elif config.peft_method == "ia3":
        peft_config = IA3Config(task_type=task_type)
    elif config.peft_method == "adalora":
            peft_config = AdaLoraConfig(
            task_type=task_type,
            target_modules=target_modules,
            init_r=12,
            target_r=8,
            tinit=max(1, int(0.1 * total_steps)),
            tfinal=max(2, int(0.8 * total_steps)),
            deltaT=1,
            total_step=total_steps,
        )
    elif config.peft_method == "prefix":
        peft_config = PrefixTuningConfig(
            task_type=task_type,
            num_virtual_tokens=20,
        )
    elif config.peft_method == "prompt":
        peft_config = PromptTuningConfig(
            task_type=task_type,
            num_virtual_tokens=20,
        )
    else:
        raise ValueError(f"Unsupported PEFT method: {config.peft_method}")

    model = get_peft_model(model, peft_config)
    return model

#list all parameters and whether they’re frozen
def print_trainable_parameters(model):
    for name, param in model.named_parameters():
        status = "TRAINABLE" if param.requires_grad else "FROZEN"
        print(f"{name:60} → {status}")

def find_classifier(model):
    for name, module in model.named_modules():
        if "classifier" in name or "score" in name or "classification_head" in name:
            return module
    return None



In [101]:

# data.py
def load_and_prepare_data(config: ExperimentConfig):
    # Load the Data from Huggingface HUb
    raw = load_dataset(config.dataset_name)

    #Take sample Data for quick testing, shuffle with seed for reproducibility
    dataset = DatasetDict({
        "train": raw["train"].shuffle(seed=config.seed).select(range(1200)),
        "test": raw["test"].shuffle(seed=config.seed).select(range(400)),
    })

    # Load the correct tokenizer based on the model.
    tokenizer = AutoTokenizer.from_pretrained(config.model_name)

    def tokenize_fn(examples):
        return tokenizer(
            examples["text"],
            truncation=True, #If the text is longer than max_length, it gets cut:
            padding=False, 
            max_length=config.max_length, #If the text is longer than max_length, it gets cut:
        )

    tokenized = dataset.map(tokenize_fn, batched=True)
    tokenized = tokenized.rename_column("label", "labels") #Hugging Face models expect the target column to be named labels. 
                                                            # This is required for trainer compatibility.
    tokenized.set_format("torch", columns=["input_ids", "attention_mask", "labels"]) #Only keep these columns and convert them to PyTorch tensors

    return tokenized, tokenizer



In [102]:
# training.py

def set_seed(seed: int):
    torch.manual_seed(seed)
    random.seed(seed)
    np.random.seed(seed)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = logits.argmax(-1)
    return {
        "accuracy": accuracy_score(labels, preds),
        "f1": f1_score(labels, preds),
    }

def build_trainer(model, dataset, config: ExperimentConfig, tokenizer):
    training_args = TrainingArguments(
        output_dir=config.output_dir(),
        learning_rate=config.learning_rate,
        per_device_train_batch_size=config.batch_size,
        num_train_epochs=config.num_epochs,
        # logging_dir=f"{config.output_dir()}/logs",
        logging_strategy="epoch",
        eval_strategy="epoch",
        save_strategy="epoch",
        load_best_model_at_end=True,
        metric_for_best_model="eval_loss",
        greater_is_better=False,
        save_total_limit=2,
        lr_scheduler_type="cosine",
        warmup_steps=100,
        report_to=["tensorboard"],
        # Optimize for CPU
        dataloader_pin_memory=False,
    )

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=dataset["train"],
        eval_dataset=dataset["test"],
        compute_metrics=compute_metrics,
        data_collator=DataCollatorWithPadding(tokenizer)
    )


    return trainer



In [103]:
 # reporting.py


def count_trainable_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

def plot_confusion_matrix(labels, preds, path_prefix):
    cm = confusion_matrix(labels, preds)
    plt.figure(figsize=(4, 3))
    sns.heatmap(
        cm, annot=True, fmt="d", cmap="Blues",
        xticklabels=["Neg", "Pos"],
        yticklabels=["Neg", "Pos"],
    )
    plt.title("Confusion Matrix")
    plt.xlabel("Predicted")
    plt.ylabel("True")
    plt.tight_layout()
    plt.savefig(f"{path_prefix}_confusion_matrix.png")
    plt.close()

def plot_loss_curve(log_history, path_prefix):
    steps = [x["step"] for x in log_history if "loss" in x]
    losses = [x["loss"] for x in log_history if "loss" in x]

    plt.figure(figsize=(5, 3))
    plt.plot(steps, losses, marker="o")
    plt.title("Training Loss Curve")
    plt.xlabel("Step")
    plt.ylabel("Loss")
    plt.tight_layout()
    plt.savefig(f"{path_prefix}_loss_curve.png")
    plt.close()

def plot_epoch_metrics(log_history, path_prefix):
    epochs = [x["epoch"] for x in log_history if "eval_accuracy" in x]
    accs = [x["eval_accuracy"] for x in log_history if "eval_accuracy" in x]
    f1s = [x["eval_f1"] for x in log_history if "eval_f1" in x]

    if not epochs:
        return

    plt.figure(figsize=(5, 3))
    plt.plot(epochs, accs, marker="o", label="Accuracy")
    plt.plot(epochs, f1s, marker="s", label="F1")
    plt.title("Per-Epoch Metrics")
    plt.xlabel("Epoch")
    plt.ylabel("Metric")
    plt.legend()
    plt.tight_layout()
    plt.savefig(f"{path_prefix}_epoch_metrics.png")
    plt.close()

def export_experiment(config, trainer, model, tokenizer, train_runtime):

    # 1. Create main experiment folder
    exp_dir = config.output_dir()
    os.makedirs(exp_dir, exist_ok=True)

    # 2. Create dashboard folder
    dashboard_dir = os.path.join(exp_dir, "dashboard")
    os.makedirs(dashboard_dir, exist_ok=True)

    # 3. Prefix for plots
    path_prefix = os.path.join(dashboard_dir, "plot")

    # 4. Generate predictions
    preds_output = trainer.predict(trainer.eval_dataset)
    preds = preds_output.predictions.argmax(axis=-1)
    labels = preds_output.label_ids

    # 5. Generate plots
    plot_confusion_matrix(labels, preds, path_prefix)
    plot_loss_curve(trainer.state.log_history, path_prefix)
    plot_epoch_metrics(trainer.state.log_history, path_prefix)

    # 6. Collect metrics
    metrics = trainer.evaluate()

    # 7. Build experiment summary
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total_params = sum(p.numel() for p in model.parameters())

    experiment_data = {
        "experiment_name": config.experiment_name(),
        "timestamp": datetime.now().isoformat(),
        "device": str(config.device()),
        "model_name": config.model_name,
        "peft_method": config.peft_method,
        "trainable_params": trainable_params,
        "trainable_pct": (trainable_params / total_params) * 100,
        "total_params": total_params,
        "metrics": metrics,
        "config": config.to_dict(),
        "best_checkpoint": trainer.state.best_model_checkpoint,
    }

    # 8. Save JSON summary
    summary_path = os.path.join(exp_dir, "experiment_summary.json")
    with open(summary_path, "w") as f:
        json.dump(experiment_data, f, indent=4)

    # 9. Save model + tokenizer
    # model.save_pretrained(exp_dir)
    if config.peft_method == "freeze":
        # full HF model (no adapters)
        model.save_pretrained(exp_dir)
    else:
        # PEFT: save only adapters
        model.save_pretrained(exp_dir, save_adapter=True)


    tokenizer.save_pretrained(exp_dir)

    print(f"Experiment saved at: {summary_path}")

In [104]:
# run_experiment.py

config = ExperimentConfig(
            model_name="bert-base-uncased",   # or "roberta-base", "distilbert-base-uncased"
            peft_method="ia3",               # lora | ia3 | adalora | prefix | prompt | freeze
        ) 


        

set_seed(config.seed)

dataset, tokenizer = load_and_prepare_data(config)
total_steps = (len(dataset["train"]) // config.batch_size) * config.num_epochs
model = build_model(config, total_steps)
print_trainable_parameters(model)
trainer = build_trainer(model, dataset, config,tokenizer)
start = time.time()
trainer.train()
runtime = time.time() - start

print("Best checkpoint:", trainer.state.best_model_checkpoint)
export_experiment(config, trainer, model, tokenizer,runtime)



Loading weights: 100%|██████████| 199/199 [00:02<00:00, 86.02it/s, Materializing param=bert.pooler.dense.weight]                                
BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those pa

base_model.model.bert.embeddings.word_embeddings.weight      → FROZEN
base_model.model.bert.embeddings.position_embeddings.weight  → FROZEN
base_model.model.bert.embeddings.token_type_embeddings.weight → FROZEN
base_model.model.bert.embeddings.LayerNorm.weight            → FROZEN
base_model.model.bert.embeddings.LayerNorm.bias              → FROZEN
base_model.model.bert.encoder.layer.0.attention.self.query.weight → FROZEN
base_model.model.bert.encoder.layer.0.attention.self.query.bias → FROZEN
base_model.model.bert.encoder.layer.0.attention.self.key.base_layer.weight → FROZEN
base_model.model.bert.encoder.layer.0.attention.self.key.base_layer.bias → FROZEN
base_model.model.bert.encoder.layer.0.attention.self.key.ia3_l.default → TRAINABLE
base_model.model.bert.encoder.layer.0.attention.self.value.base_layer.weight → FROZEN
base_model.model.bert.encoder.layer.0.attention.self.value.base_layer.bias → FROZEN
base_model.model.bert.encoder.layer.0.attention.self.value.ia3_l.default → TRAINAB

Epoch,Training Loss,Validation Loss,Accuracy,F1
1,0.694955,0.680576,0.562500,0.682396


Best checkpoint: ./outputs/ia3_20260310_165735\checkpoint-75


Experiment saved at: ./outputs/ia3_20260310_165735\experiment_summary.json


In [ ]:
import os
import json
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from peft import AutoPeftModelForSequenceClassification

def load_experiment(experiment_dir: str):
    # load summary
    summary_path = os.path.join(experiment_dir, "experiment_summary.json")

    if not os.path.exists(summary_path):
        raise FileNotFoundError(f"Could not find summary at {summary_path}")
    
    with open(summary_path, "r") as f:
        summary = json.load(f)

    peft_method = summary["peft_method"]

    print(f"Loading tokenizer from {experiment_dir}...")
    tokenizer = AutoTokenizer.from_pretrained(experiment_dir)

    adapter_config = os.path.join(experiment_dir, "adapter_config.json")
    full_model = os.path.join(experiment_dir, "pytorch_model.bin")

    if os.path.exists(adapter_config) and not os.path.exists(full_model):
        print(f"🚀 Detected PEFT model ({peft_method}). Loading adapters...")
        model = AutoPeftModelForSequenceClassification.from_pretrained(
            experiment_dir,
            device_map="auto",
        )
    else:
        print("Loading full HF model (freeze or full fine-tune)...")
        model = AutoModelForSequenceClassification.from_pretrained(
            experiment_dir,
            device_map="auto",
        )

    model.eval()
    return model, tokenizer, summary


def predict_text(model, tokenizer, text: str):
    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=128,
    ).to(model.device)

    with torch.no_grad():
        outputs = model(**inputs)
        logits = outputs.logits
        probs = torch.softmax(logits, dim=-1)
        pred = torch.argmax(probs, dim=-1).item()

    return {
        "prediction": pred,
        "probabilities": probs.squeeze().tolist(),
        "all_probs": probs.squeeze().tolist(),
    }

# freeze_20260307_215157\experiment_summary.json
#adalora_20260307_221806\checkpoint-38
if __name__ == "__main__":
    experiment_dir = "./outputs/adalora_20260307_221806"  # or freeze_...

    model, tokenizer, summary = load_experiment(experiment_dir)

    text = "The movie was absolutely fantastic!"
    result = predict_text(model, tokenizer, text)

    print("Input:", text)
    print("Predicted label:", result["prediction"])
    print("Probabilities:", result["probabilities"])

Loading tokenizer from ./outputs/adalora_20260307_221806...
🚀 Detected PEFT model (adalora). Loading adapters...


Loading weights: 100%|██████████| 199/199 [00:01<00:00, 150.28it/s, Materializing param=bert.pooler.dense.weight]                               
BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those pa

Input: The movie was absolutely fantastic!
Predicted label: 1
Probabilities: [0.48584890365600586, 0.5141510963439941]
